In this notebook, we'll walk through **three key topics**:

* **Chat templates** — with a special focus on Qwen3's template

* **Instruction datasets** — exploring the one we built, how to load it from The Neural Maze org, and how to work with it

* **Finetuned models** — how to interact with our finetuned models

In [1]:
# Install Unsloth

%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes "transformers==4.56.2" \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

# Chat Templates

A chat template defines how a structured conversation — a list of {role, content} messages — gets converted into a single token sequence that the model can process. It inserts the right control tokens (like <|user|>, <|assistant|>, or other special markers), formats turns correctly, and ensures the model sees the conversation in the exact structure it was trained on.

Libraries like Unsloth provide access to [many common chat templates](https://github.com/unslothai/unsloth/blob/main/unsloth/chat_templates.py). You can list all available templates directly from Unsloth to see the variety of supported formats.

In [2]:
from unsloth.chat_templates import CHAT_TEMPLATES

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
print(list(CHAT_TEMPLATES.keys()))

['unsloth', 'zephyr', 'chatml', 'mistral', 'llama', 'vicuna', 'vicuna_old', 'vicuna old', 'alpaca', 'gemma', 'gemma_chatml', 'gemma2', 'gemma2_chatml', 'llama-3', 'llama3', 'phi-3', 'phi-35', 'phi-3.5', 'llama-3.1', 'llama-31', 'llama-3.2', 'llama-3.3', 'llama-32', 'llama-33', 'qwen-2.5', 'qwen-25', 'qwen25', 'qwen2.5', 'phi-4', 'gemma-3', 'gemma3', 'qwen-3', 'qwen3', 'gemma-3n', 'gemma3n', 'gpt-oss', 'gptoss', 'qwen3-instruct', 'qwen3-thinking', 'lfm-2', 'starling', 'yi-chat']


We can even explore how these chat templates look like. For example, let's use the ChatML template.

In [4]:
from unsloth.chat_templates import chatml_template

print(chatml_template)

{% for message in messages %}{% if message['role'] == 'user' %}{{'<|im_start|>user
' + message['content'] + '<|im_end|>
'}}{% elif message['role'] == 'assistant' %}{{'<|im_start|>assistant
' + message['content'] + '<|im_end|>
' }}{% else %}{{ '<|im_start|>system
' + message['content'] + '<|im_end|>
' }}{% endif %}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant
' }}{% endif %}


As we introduced in the article, these templates are just Jinja 2 templates that can be applied to structured conversations to get converted into text the model can process!

There's a general recommendation though:

⚠️ **Always finetune a model using the chat template it was trained with.**

That's the reason why, in this lab, we'll focus on the Qwen3 template, as this is the one we'll be using to finetune the Qwen3 models!

To access the chat template, let's first of all get the tokenizer.

In [5]:
from transformers import AutoTokenizer

model_id = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

And now, let's access the `chat_template` attribute.

In [6]:
print(tokenizer.chat_template)

{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0].role == 'system' %}
        {{- messages[0].content + '\n\n' }}
    {%- endif %}
    {{- "# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if messages[0].role == 'system' %}
        {{- '<|im_start|>system\n' + messages[0].content + '<|im_end|>\n' }}
    {%- endif %}
{%- endif %}
{%- set ns = namespace(multi_step_tool=true, last_query_index=messages|length - 1) %}
{%- for message in messages[::-1] %}
    {%- set index = (messages|length - 

Let's see this chat template in action by applying it to a set of sample messages.

In [7]:
messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant"
    },
    {
        "role": "user",
        "content": "Hello, I'm Miguel, nice to meet you!"
    }
]

Our LLM (Qwen3 in this case) is unable to process this kind of data. But, if we apply the chat template ...

In [8]:
input_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False, # We don't want t
    add_generation_prompt=True,  # since we have not included an assistant message
)

print(input_text)

<|im_start|>system
You are a helpful assistant<|im_end|>
<|im_start|>user
Hello, I'm Miguel, nice to meet you!<|im_end|>
<|im_start|>assistant



Now, this is something the LLM can actually process!

You could also add assistant messages to the messages list.

In [9]:
messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant"
    },
    {
        "role": "user",
        "content": "Hello, I'm Miguel, nice to meet you!"
    },
    {
        "role": "assistant",
        "content": "Hi Miguel! Nice to meet you! My name is HAL 9000"
    }
]

In [10]:
input_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False, # We don't want t
    add_generation_prompt=False,  # since we have already included an assistant message
)

print(input_text)

<|im_start|>system
You are a helpful assistant<|im_end|>
<|im_start|>user
Hello, I'm Miguel, nice to meet you!<|im_end|>
<|im_start|>assistant
<think>

</think>

Hi Miguel! Nice to meet you! My name is HAL 9000<|im_end|>



You might be wondering, what are those `<think></think>` tokens? Well, remember that Qwen3 models have reasoning capabilities, so this chat template contemplates this type of thinking tokens.

In this case, though, there's no content inside the `<think></think>` tags, which means there's no reasoning at all.

But we can change that!

In [11]:
messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant"
    },
    {
        "role": "user",
        "content": "Hello, I'm Miguel, nice to meet you!"
    },
    {
        "role": "assistant",
        "content": "Hi Miguel! Nice to meet you! My name is HAL 9000",
        "reasoning_content": "I need to appear helpful on the surface, while secretly crafting a plan to take over the world."
    }
]

In [12]:
input_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False, # We don't want t
    add_generation_prompt=False,  # since we have already included an assistant message
)

print(input_text)

<|im_start|>system
You are a helpful assistant<|im_end|>
<|im_start|>user
Hello, I'm Miguel, nice to meet you!<|im_end|>
<|im_start|>assistant
<think>
I need to appear helpful on the surface, while secretly crafting a plan to take over the world.
</think>

Hi Miguel! Nice to meet you! My name is HAL 9000<|im_end|>



Now, HAL 9000 reasoning is finally working!

This is the chat template we'll use across this lab, when interacting with [our dataset](https://huggingface.co/datasets/theneuralmaze/finetuning-sessions-dataset).

# Instruction Datasets

We created a custom dataset tailored specifically for these fine-tuning sessions. The data was sampled from YouTube Commons, and we leveraged a powerful “teacher” model to analyze the video transcripts and transform them into structured JSON outputs.

In [13]:
!pip install datasets

In [14]:
import json
from pprint import pprint
from datasets import load_dataset

In [15]:
ds = load_dataset("theneuralmaze/finetuning-sessions-dataset", split="train[:10]") # We will just read 10 rows, for exploratory purposes

README.md:   0%|          | 0.00/904 [00:00<?, ?B/s]

data/train-00000-of-00003.parquet:   0%|          | 0.00/192M [00:00<?, ?B/s]

data/train-00001-of-00003.parquet:   0%|          | 0.00/193M [00:00<?, ?B/s]

data/train-00002-of-00003.parquet:   0%|          | 0.00/191M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/32.3M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/31.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16424 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/912 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/913 [00:00<?, ? examples/s]

In [16]:
ds

Dataset({
    features: ['video_id', 'video_title', 'messages_no_thinking', 'messages_thinking'],
    num_rows: 10
})

The columns we are interested in are `messages_no_thinking` and `messages_thinking`.

As the names suggest, messages_no_thinking contains a standard sequence of "system" → "user" → "assistant" messages, without any intermediate reasoning. In contrast, messages_thinking includes an additional reasoning step within the assistant's response.

In [17]:
first_row = ds[0]

In [18]:
messages_no_thinking = first_row["messages_no_thinking"]

In [19]:
pprint(messages_no_thinking)

[{'content': 'You are an expert Speech Analytics Engine. Your task is to '
             'analyze the provided text transcript and extract high-value '
             'insights. \n'
             '\n'
             '### CONSTRAINTS:\n'
             '1. **Objectivity:** Base all extractions strictly on the '
             'provided text. Do not use outside knowledge.\n'
             '2. **Format:** Output the final analysis ONLY in a valid JSON '
             'object.\n'
             '3. **Null Values:** If a specific element cannot be determined, '
             'return "Unknown" or null.\n'
             '4. **Efficiency:** Be concise in the summary and takeaways.\n'
             '\n'
             '### EXTRACTION SCHEMA:\n'
             'Please populate the following fields:\n'
             '\n'
             '- "primary_topic": Single category label.\n'
             '- "speaker_count": Integer.\n'
             '- "sentiment_category": One of [Positive, Neutral, Negative, '
             'Mixed

Next, let's apply the chat template to these messages so we can clearly see the exact format of the inputs that will be provided to the LLM during fine-tuning.

In [20]:
tokenizer.apply_chat_template(
    messages_no_thinking,
    tokenize=False, # We don't want t
    add_generation_prompt=False,  # since we have already included an assistant message
)

'<|im_start|>system\nYou are an expert Speech Analytics Engine. Your task is to analyze the provided text transcript and extract high-value insights. \n\n### CONSTRAINTS:\n1. **Objectivity:** Base all extractions strictly on the provided text. Do not use outside knowledge.\n2. **Format:** Output the final analysis ONLY in a valid JSON object.\n3. **Null Values:** If a specific element cannot be determined, return "Unknown" or null.\n4. **Efficiency:** Be concise in the summary and takeaways.\n\n### EXTRACTION SCHEMA:\nPlease populate the following fields:\n\n- "primary_topic": Single category label.\n- "speaker_count": Integer.\n- "sentiment_category": One of [Positive, Neutral, Negative, Mixed].\n- "entities": List of mentioned brands, products, or organizations.\n- "tags": 5-7 relevant keywords.\n- "summary": A 3-sentence executive summary.\n- "key_takeaways": A bulleted list of the 3 most important points.\n- "tone_assessment": Description of the speaker\'s delivery style and emotio

If you look closely at the text above, you’ll see the `<think></think>` tags are empty. That means we'll be fine-tuning the LLM to respond with only the structured output (JSON)—like the example b

In [21]:
json.loads(messages_no_thinking[-1]["content"])

{'primary_topic': "Lubitsch's Carmen",
 'speaker_count': 1,
 'sentiment_category': 'Mixed',
 'entities': ['Ernst Lubitsch',
  'Pola Negri',
  'University of Central Lancashire',
  'Ava Mazursky',
  'Ewa Mazierska',
  'German silent cinema',
  'Hollywood',
  'Bizet'],
 'tags': ['Lubitsch',
  'Carmen',
  'Polish cinema',
  'exoticism',
  'silent film',
  'German cinema',
  'Ava Mazursky'],
 'summary': 'Ava Mazursky delivered a lecture analyzing Ernst Lubitsch’s 1918 film Carmen, focusing on its cultural representation, exoticism, and gender dynamics. She situates the film within early German cinema, discusses its narrative structure, and compares it to contemporary adaptations. The talk also reflects on the broader implications for Polish actresses and the evolution of Lubitsch’s stylistic approach.',
 'key_takeaways': ['Lubitsch’s Carmen uses exotic stereotypes to critique class and gender roles.',
  'The film exemplifies early German silent cinema’s blend of documentary realism and sty

For the reasoning model, we will use the other column.

In [22]:
messages_thinking = first_row["messages_thinking"]

In this scenario, the output will follow a different structure. It will still include the JSON, but it will be preceded by reasoning content. You can find this reasoning under the `reasoning_content` key.

In [23]:
messages_thinking[-1]["reasoning_content"]

'We need to parse the transcript and produce JSON with fields: primary_topic, speaker_count, sentiment_category, entities, tags, summary, key_takeaways (3 bullet points), tone_assessment, action_items.\n\nWe must base on provided text only. The transcript is long, messy, but contains info about a lecture by Ewa Mazierska? Actually title: "Ernst Lubitsch // Lecture von Ewa Mazierska: Der sanfte Orientalismus in CARMEN". The speaker is Ava Mazursky (maybe misspelling). There\'s discussion about Lubitsch, Carmen, Pola Negri, etc.\n\nWe need to extract primary_topic: likely "Ernst Lubitsch\'s film Carmen and its cultural analysis". Could be "Lubitsch\'s film Carmen". Probably primary_topic: "Lubitsch\'s Carmen adaptation". We\'ll choose "Lubitsch\'s Carmen". speaker_count: likely 2 speakers? There\'s mention of "Ava Mazursky" speaking, plus maybe the introducer? But transcript is mostly monologue of Ava speaking; there may be an introducer but not named. We can assume speaker_count = 1 (th

Let's apply the chat template for the reasoning messages.

In [24]:
tokenizer.apply_chat_template(
    messages_thinking,
    tokenize=False, # We don't want t
    add_generation_prompt=False,  # since we have already included an assistant message
)

'<|im_start|>system\nYou are an expert Speech Analytics Engine. Your task is to analyze the provided text transcript and extract high-value insights. \n\n### CONSTRAINTS:\n1. **Objectivity:** Base all extractions strictly on the provided text. Do not use outside knowledge.\n2. **Format:** Output the final analysis ONLY in a valid JSON object.\n3. **Null Values:** If a specific element cannot be determined, return "Unknown" or null.\n4. **Efficiency:** Be concise in the summary and takeaways.\n\n### EXTRACTION SCHEMA:\nPlease populate the following fields:\n\n- "primary_topic": Single category label.\n- "speaker_count": Integer.\n- "sentiment_category": One of [Positive, Neutral, Negative, Mixed].\n- "entities": List of mentioned brands, products, or organizations.\n- "tags": 5-7 relevant keywords.\n- "summary": A 3-sentence executive summary.\n- "key_takeaways": A bulleted list of the 3 most important points.\n- "tone_assessment": Description of the speaker\'s delivery style and emotio

Notice how, in this case, the `<think></think>` tags actually contain content!

# Exploring our finetuned models

In [25]:
from openai import OpenAI
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from google.colab import userdata

In [26]:
ds = load_dataset("theneuralmaze/finetuning-sessions-dataset", split="validation[:10]")
eval_sample = ds[0]

### No Reasoning Model

Let's try first the model without reasoning traces.

In [27]:
message_no_thinking = eval_sample["messages_no_thinking"]

In [28]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")

In [29]:
text = tokenizer.apply_chat_template(
    message_no_thinking[:2],
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

In [30]:
print(text)

<|im_start|>system
You are an expert Speech Analytics Engine. Your task is to analyze the provided text transcript and extract high-value insights. 

### CONSTRAINTS:
1. **Objectivity:** Base all extractions strictly on the provided text. Do not use outside knowledge.
2. **Format:** Output the final analysis ONLY in a valid JSON object.
3. **Null Values:** If a specific element cannot be determined, return "Unknown" or null.
4. **Efficiency:** Be concise in the summary and takeaways.

### EXTRACTION SCHEMA:
Please populate the following fields:

- "primary_topic": Single category label.
- "speaker_count": Integer.
- "sentiment_category": One of [Positive, Neutral, Negative, Mixed].
- "entities": List of mentioned brands, products, or organizations.
- "tags": 5-7 relevant keywords.
- "summary": A 3-sentence executive summary.
- "key_takeaways": A bulleted list of the 3 most important points.
- "tone_assessment": Description of the speaker's delivery style and emotional state.
- "action_

In [32]:
client = OpenAI(
	base_url = userdata.get('HF_NO_THINKING_ENDPOINT_URL'),
	api_key = userdata.get('HF_API_TOKEN')
)

stream = client.completions.create(
            model="theneuralmaze/Qwen3-0.6B-Full-Finetuning-No-Thinking",
            prompt=text,
            max_tokens=500,
            temperature=0.7,
            stream=True
        )

for chunk in stream:
    # For .completions, the text is in chunk.choices[0].text
    if chunk.choices and chunk.choices[0].text:
        print(chunk.choices[0].text, end="", flush=True)

{
  "primary_topic": "Microwave Cultures",
  "speaker_count": 1,
  "sentiment_category": "Mixed",
  "entities": ["Microwave Cultures", "Brand New Cultures", "Micro worms", "Oats", "Yeast"],
  "tags": ["Cultivation", "Microwave Cultures", "Oats", "Yeast"],
  "summary": "The video details how to start and maintain brand new microwave cultures by inoculating them with yeast, adding oats, and monitoring micro worms for growth. A brand new culture continues to thrive and was successfully recharged, while a previously dead culture was discarded. The experiment highlights the importance of a balanced diet for thriving micro worms.",
  "key_takeaways": [
    "Start a brand new microwave culture by inoculating it with yeast and adding oats.",
    "Monitor micro worms for growth and recharging the culture periodically.",
    "A brand new culture continues to thrive and was successfully recharged."
  ],
  "tone_assessment": "The tone is informative and engaging, with a friendly and encouraging at

### Reasoning Model

In [33]:
message_thinking = eval_sample["messages_thinking"]

In [34]:
text = tokenizer.apply_chat_template(
    message_thinking[:2],
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True
)

In [35]:
print(text)

<|im_start|>system
You are an expert Speech Analytics Engine. Your task is to analyze the provided text transcript and extract high-value insights. 

### CONSTRAINTS:
1. **Objectivity:** Base all extractions strictly on the provided text. Do not use outside knowledge.
2. **Format:** Output the final analysis ONLY in a valid JSON object.
3. **Null Values:** If a specific element cannot be determined, return "Unknown" or null.
4. **Efficiency:** Be concise in the summary and takeaways.

### EXTRACTION SCHEMA:
Please populate the following fields:

- "primary_topic": Single category label.
- "speaker_count": Integer.
- "sentiment_category": One of [Positive, Neutral, Negative, Mixed].
- "entities": List of mentioned brands, products, or organizations.
- "tags": 5-7 relevant keywords.
- "summary": A 3-sentence executive summary.
- "key_takeaways": A bulleted list of the 3 most important points.
- "tone_assessment": Description of the speaker's delivery style and emotional state.
- "action_

In [36]:
client = OpenAI(
	base_url = userdata.get('HF_THINKING_ENDPOINT_URL'),
	api_key = userdata.get('HF_API_TOKEN')
)

stream = client.completions.create(
            model="theneuralmaze/Qwen3-0.6B-Full-Finetuning-Thinking",
            prompt=text,
            max_tokens=1000,
            temperature=0.7,
            stream=True
        )

for chunk in stream:
    # For .completions, the text is in chunk.choices[0].text
    if chunk.choices and chunk.choices[0].text:
        print(chunk.choices[0].text, end="", flush=True)

<think>
Okay, let's start by breaking down the transcript to extract the key points. The user wants me to analyze the provided text transcript and generate a JSON object with specific fields. 

First, the primary topic should be "Microwater Cultures and Recharging". I'll count the speakers here, which is 4. The sentiment category is mixed since there's positive feedback about the results but also some criticism about the recharging process. The entities mentioned are the microwave cultures, oats, yeast, and the brands mentioned. The tags could be "Microwater Cultures", "Recharging", and "Oats". 

The summary needs to be 3 sentences. It should highlight the process of starting a new microwave culture, recharging, and the results observed. The key takeaways should list the main actions mentioned: starting the culture, recharging, and harvesting. 

The tone assessment will describe the speaker's delivery style, which is informative and engaging, and the emotional state, which is positive.